# IsoKernel Pipeline Demonstration
This notebook demonstrates the execution of the IsoKernel knowledge engineering pipeline.

The pipeline comprises two major phases:
1. **Phase 1: Schema-Less Triple Extraction** - Using an LLM to extract Subject-Predicate-Object triples from raw text.
2. **Phase 2: Refinement & Logic Clustering** - Using graph topology (Louvain/Leiden) and semantic embeddings to cluster entities and reduce noise.


In [1]:
import os
from IPython.display import IFrame
import yaml

from src.core.models import DocumentSource
from src.orchestrator.pipeline import PipelineOrchestrator

# Ensure config points to correct input/output directories
config_path = "config.yaml"

print("Current configuration for the pipeline:")
with open(config_path, 'r') as f:
    print(f.read())


orchestrator = PipelineOrchestrator(config_path=config_path, domain="knowledge")
#orchestrator = PipelineOrchestrator(config_path=config_path, domain="clinical child psychiatry")

Current configuration for the pipeline:
# IsoKernel Master Configuration

pipeline:
  run_phase_1: true
  run_phase_2: true
  run_phase_4: true
  max_concurrent_llm_calls: 4
  
extraction:
  domain: "general knowledge"
  max_retries: 3
  theme_chunk_max_words: 4000
  triple_chunk_max_words: 1000
  
refinement:
  # Semantic Embedding Parameters
  use_embeddings: true
  embedding_model: "all-MiniLM-L6-v2"
  clustering_method: "agglomerative"  # or "hdbscan"
  # similarity_threshold: Minimum cosine similarity measure required (0.0 to 1.0) to map two string variants together. Higher means stricter mapping.
  similarity_threshold: 0.8
  
  # compression_mode: 
  # - "isolated": Clusters and deduplicates strings within their own specific silo (subjects with subjects, predicates with predicates).
  # - "unified": Lumps everything together into a giant pool before measuring semantic distance.
  compression_mode: "isolated"
  
  # compress_fields: Choose exactly which fields across the triplet 

## Executing the Pipeline Step-by-Step

Instead of running the entire pipeline at once natively (`orchestrator.run(folder_path)`), we can execute it step-by-step for better diagnostic review. First, we prepare the documents.

In [2]:
#folder_path = "./data/inputs/testdocs"
folder_path = "./data/inputs/philosophy_texts"
documents = orchestrator.prepare_documents([folder_path])
print(f"Loaded {len(documents)} documents for processing.")

Loaded 2 documents for processing.


### Phase 1: Foundational Knowledge Extraction

To maximize schema determinism, we now use a mathematical Two-Pass approach to extract facts. 

**Pass A: Theme Discovery** reads the document strictly as an "Ontological Concept Engine" to dynamically extract the overarching topics first.
**Pass B: Schema-Mapped Discovery** then extracts fine-grained triples and mathematically forces them to structurally align with the Themes discovered in Pass A.

#### Step 1.1: Thematic Discovery
Let's query the LLM to structurally deduce the 5 most critical overarching themes from the text.

In [3]:
theme_maps = orchestrator.extract_themes(documents)
print(f"Discovered themes mapped across {len(theme_maps)} documents.")

# Preview the themes discovered in the first document functionally:
import json
first_doc = list(theme_maps.keys())[0]
print(f"Themes for {first_doc}:")
print(json.dumps(theme_maps[first_doc], indent=2))

2026-04-12 16:16:25,999 | phase1_extractor | INFO | Configuring local LLM provider via AsyncOpenAI API schema...
2026-04-12 16:16:26,153 | phase1_extractor | INFO | Initialized TripleExtractor for domain 'knowledge' using model 'mistral-nemo:12b-instruct-2407-q4_K_M' via 'local'
2026-04-12 16:16:26,156 | orchestrator | INFO | Discovering themes for chunk: metaphysics.txt_chunk_0
2026-04-12 16:16:26,662 | orchestrator | INFO | Discovering themes for chunk: metaphysics.txt_chunk_1
2026-04-12 16:16:27,162 | orchestrator | INFO | Discovering themes for chunk: metaphysics.txt_chunk_2
2026-04-12 16:16:27,662 | orchestrator | INFO | Discovering themes for chunk: metaphysics.txt_chunk_3
2026-04-12 16:18:00,370 | orchestrator | INFO | Discovering themes for chunk: schopenhauer.txt_chunk_0
2026-04-12 16:18:31,429 | orchestrator | INFO | Discovering themes for chunk: schopenhauer.txt_chunk_1
2026-04-12 16:18:43,819 | orchestrator | INFO | Discovering themes for chunk: schopenhauer.txt_chunk_2


Discovered themes mapped across 2 documents.
Themes for metaphysics.txt:
[
  "Scientific Research Methodology",
  "Metaphysical Unity",
  "Abstract Entities vs Real Entities",
  "Metaphysical Anti-Realism",
  "Text Processing Assistance",
  "Modal Realism",
  "Identity and Change Over Time",
  "Epistemological Methodology",
  "Metaphysical Modality",
  "Social Metaphysics",
  "Planetary Science",
  "Epistemic Justification",
  "Extraterrestrial Life Possibilities"
]


#### Step 1.2: Global Ontology Synthesis
To ensure overlapping themes discovered across different training documents do not create drift, this step consolidates all uniquely found Document-Level themes into a single formal *Master Corpus Ontology*.

In [4]:
master_themes = orchestrator.consolidate_themes(theme_maps)
print(f"Consolidated into {len(master_themes)} Master Themes scaling natively across all documents:\n")
for t in master_themes:
    print(f"- {t}")

2026-04-12 16:22:51,837 | phase1_extractor | INFO | Configuring local LLM provider via AsyncOpenAI API schema...
2026-04-12 16:22:51,873 | phase1_extractor | INFO | Initialized TripleExtractor for domain 'knowledge' using model 'mistral-nemo:12b-instruct-2407-q4_K_M' via 'local'
2026-04-12 16:22:51,873 | orchestrator | INFO | Consolidating 25 document-level themes into a Master Corpus Ontology...


Consolidated into 9 Master Themes scaling natively across all documents:

- Metaphysics
- Epistemology
- Philosophical Movements, Figures, and History
- Ethics and Axiology
- Aesthetics
- Cosmology and Planetary Science
- Existentialism
- Text Processing and Analysis
- Social Metaphysics


#### Step 1.3: Constrained Triplet Mapping
Now, pipe the structured ontological themes into the final triplet extraction sequence mathematically bounding the logic.

In [ ]:
triples = orchestrator.extract_triples(documents, master_themes)
print(f"Extracted {len(triples)} refined schema-bound triples.")

# Review a sample of the extracted triples natively mapping theme_association properties:
print(json.dumps(triples[:2], indent=2))

2026-04-12 16:24:38,225 | phase1_extractor | INFO | Configuring local LLM provider via AsyncOpenAI API schema...
2026-04-12 16:24:38,261 | phase1_extractor | INFO | Initialized TripleExtractor for domain 'knowledge' using model 'mistral-nemo:12b-instruct-2407-q4_K_M' via 'local'
2026-04-12 16:24:38,264 | orchestrator | INFO | Processing structural extraction for chunk: metaphysics.txt_chunk_0
2026-04-12 16:24:38,768 | orchestrator | INFO | Processing structural extraction for chunk: metaphysics.txt_chunk_1
2026-04-12 16:24:39,268 | orchestrator | INFO | Processing structural extraction for chunk: metaphysics.txt_chunk_2
2026-04-12 16:24:39,768 | orchestrator | INFO | Processing structural extraction for chunk: metaphysics.txt_chunk_3
2026-04-12 16:26:04,897 | orchestrator | INFO | Processing structural extraction for chunk: metaphysics.txt_chunk_4


### Phase 2: Topological Compression

The orchestrator now transitions into Phase 2, which transforms raw unrefined text strings extracted in Phase 1 into a clean, mathematically structured, and visually interactive Knowledge Graph. We do this by breaking Semantic Compression down into highly observable sub-processes.

#### Step 2.1: Vector Encodings & TF-IDF Gravity
This first block filters the raw extractions and inherently maps them into mathematical vector computations.

1. **Target Field Extraction**: Iterates over all raw triples and dynamically isolates text based on your `compress_fields` configuration (e.g., retrieving only "Subjects" and "Objects").
2. **Density Extraction (Gravity Weighting)**: Resolving geometric drift, it computes the absolute frequency distribution density map (`node_counts`) across all extracted entities instead of using standard `set()` deduplication. This mathematically calculates their TF-IDF Gravity.
3. **Semantic Encoding**: Initializes your Sentence Transformer (e.g., `all-MiniLM-L6-v2`) to convert every unique entity string into a dense numerical vector array (e.g., a 384-dimensional tensor).

In [ ]:
node_counts = orchestrator.extract_unique_nodes(triples)
print(f"Extracted {len(node_counts)} mathematically weighted density mappings for compression.")

embeddings = orchestrator.create_embeddings(node_counts)
print(f"Computed initial embeddings shape: {embeddings.shape if embeddings is not None else 'N/A'}")

#### Step 2.3: Frequency-Weighted Spectral Decomposition (PCA)
Language models often yield vectors embedded with massive amounts of mathematical variance (frequently called "linguistic noise").

If `use_spectral_decomposition` is enabled in your configuration, we conditionally inject Principal Component Analysis (PCA) here. This bounds the total 384 dimensions mathematically down strictly to the absolute top invariant principal eigenvectors (e.g., 12 dimensions). Shrinking parameters strictly to primary mathematical variances brutally forces strings into a tighter semantic vector space, which drastically stabilizes our geometric clustering results in the forthcoming steps.

In [ ]:
# Conditionally apply Spectral Vector Geometry ONLY if enabled
use_pca = orchestrator.config.get("refinement", {}).get("use_spectral_decomposition", False)

if use_pca:
    print("Spectral Geometry PCA is enabled. Decompressing matrices...")
    embeddings = orchestrator.apply_spectral_decomposition(node_counts, embeddings)
    print(f"New Spectral embeddings shape: {embeddings.shape if embeddings is not None else 'N/A'}")
else:
    print("Spectral Geometry PCA is disabled. Retaining initial matrices natively.")

#### Step 2.4: Agglomerative Structural Proposals (Clustering)
With strings cleanly mapped into vectors, the orchestrator begins determining which concepts belong functionally to identical "Semantic Communities."

1. **Agglomerative Proximity Mapping**: Initializes a hierarchical spatial `cosine` mapping algorithm natively.
2. **Threshold Halting**: Without requiring a preset group count, it pairs vectors together "bottom-up" over an average linkage continuously until your configured strict `similarity_threshold` barrier dynamically aborts the grouping logically.
3. **Associative Schema Compilation**: The system then inverses this map, returning an explicit dictionary where key cluster IDs directly govern arrays tracking every text variation mapped natively to that semantic grouping.

In [ ]:
clusters = orchestrator.compute_clusters(node_counts, embeddings)
print(f"Generated {len(clusters)} mathematical semantic clusters.")

# Let's inspect the grouped content inherently
list(clusters.items())[:2]

#### Step 2.5: Contextual Guardrails (Semantic Intercept)
Before we permanently rely on these clusters to collapse variables, we insert a strict programmatic AI validation gate. The system maps the groups' mathematical clusters and asks: *'Does merging these conceptually group them in a way that destroys critical distinguishing accuracy?'*
If True: the cluster is destroyed and entities are broken back into singletons for protection.
If False: the math is formally approved.

In [ ]:
validated_clusters = orchestrator.verify_clusters(clusters)

destroyed = len(clusters) - len([k for k, v in validated_clusters.items() if len(v) > 1])
if destroyed > 0:
    print(f"Validation Engine intervened! Mathematically rejected contextual mappings. Splitting {destroyed} groupings back apart securely...")
else:
    print(f"Contextual constraints verified seamlessly: {len(validated_clusters)} arrays successfully persisted.")

#### Step 2.6: Taxonomic Consolidation (Hypernym Resolution)
Given clustered arrays capturing semantically identical string variations (e.g., `["Patient", "Patients", "the patient"]`), this logic executes highly structured heuristics natively to select the absolute single definitive "Hypernym" representation mapping back to overwrite them all.

Using the default `"semantic_centroid"` strategy:
1. It analyzes the underlying geometric tensors exclusively matching the isolated inner members of a target cluster.
2. It averages out a comprehensive mathematical "middle point" natively acting as the absolute centroid tensor strictly representing that cluster's central meaning uniformly.
3. It conditionally transmits this centroid out to the underlying local LLM deductive logic (`instructor` bindings) attempting to structurally infer an objective abstraction ("Taxonomic Lifting", e.g., Centroid: 'Advil' -> Is-A 'Ibuprofen'). If logic fails, it blindly maps directly to the closest geometric neighbor continuously.

In [ ]:
node_mapping, cluster_logs = orchestrator.resolve_hypernyms(validated_clusters, list(node_counts.keys()), embeddings, triples)
print(f"Successfully resolved definitive taxonomic mapping for exactly {len(node_mapping)} entities out of the {len(node_counts)} mathematically weighted density mappings extracted.")

# Peek into a subset of final string translation instructions mathematically evaluated
list(node_mapping.items())[:3]

#### Step 3.1 & 3.2: Global Edge Graph Formation & Network Physics
The final segment of our modular Phase 2 execution applies our rigorously vetted taxonomies physically bridging arrays across the core visual Network.

1. **Replacement Structuring**: Safely scans through original dataset triples substituting raw extractions intelligently via the finalized `node_mapping` structures derived upstream mathematically.
2. **Graph Assembly**: Populates directional `<Subject>` -> `<Object>` vectors explicitly rendering logic continuously natively.
3. **Unsupervised Community Segmentations**: Rather than basic semantic geometries locally, this step examines macro topographical connectedness (densities mapping network connections) using deep mathematical segmentations (e.g., Louvain algorithms mapping dynamic undirected matrices inherently computing partitioned communities natively bypassing generic boundaries globally).

In [ ]:
compressed_triples = orchestrator.apply_compression(triples, node_mapping)
print(f"Compression sequence finished! Network reduced gracefully to exactly {len(compressed_triples)} uniform triplets natively.")

graph, output_html, schemas_out = orchestrator.build_and_detect_communities(compressed_triples)
print(f"Dynamic Visual Graph and Structural Data serialized uniformly.\nHTML Visualization Output: {output_html}\nCluster Tables: {schemas_out}")

## Phase 4: Schema Synthesizer
With all statistical variation completely mathematically crushed and community boundaries formally established algorithmically, we now reverse the pipeline exactly.

We pass these structural Louvain communities distinctly to the native Python generator LLM. The AI explicitly translates the Network topologies straight into perfect strictly-typed `Pydantic` schemas.py script. It logically locks the predicate fields and enforces strict isolated Enum options derived securely from graph nodes mapping natively onto disk (`data/outputs/schemas_code/schemas.py`)!

In [ ]:
written_schemas = orchestrator.synthesize_schemas(graph)

if written_schemas:
    print(f"\nSynthesizer mathematically parsed graph loops and output {len(written_schemas)} formally validated schemas natively:\n")
    for ws in written_schemas:
        print(f" -> Generated Schema: {ws}")
else:
    print("No topologies translated successfully.")

## Visualizing the Results
The orchestrator outputs interactive network graphs as HTML files in the `graph_progressions` directory. We can visualize the final clustered community graph below.

In [ ]:
# Display the generated graph dynamically nested explicitly in the notebook outputs
html_path = "./data/outputs/graphs/03_final_graph_with_communities.html"
if os.path.exists(html_path):
    from IPython.display import IFrame
    display(IFrame(src=html_path, width="100%", height="750px"))
else:
    print(f"Final interactive HTML view missing from path logs natively targeting: {html_path}")